In [1]:
# read in the data 
import pandas as pd
rivers = pd.read_csv('/Users/lisakelly/Desktop/masters/y1/project_2/progress/april/individual river imputation/machine learning/master_dataset_inputed_with_ndvi.csv')
rivers_original = rivers.copy()

# sanity check 
print(rivers.head())
print(rivers.info())
print(rivers.describe())


rivers["date"] = pd.to_datetime(rivers[["year", "month"]].assign(day=1))
rivers_original["date"] = pd.to_datetime(rivers_original[["year", "month"]].assign(day=1))

rivers = rivers.sort_values("date").set_index("date")
rivers_original = rivers_original.sort_values("date").set_index("date")

rivers.groupby("river")["water_level"].std()

     river        date  year  month  water_level  runoff  temp_max_observed  \
0  Buzimba  1981-01-01  1981      1     0.541000   16.94          26.619355   
1  Buzimba  1981-02-01  1981      2     0.485000   15.35          26.696429   
2  Buzimba  1981-03-01  1981      3     0.548387   17.96          26.087097   
3  Buzimba  1981-04-01  1981      4     0.648000   13.90          26.093333   
4  Buzimba  1981-05-01  1981      5     0.594194   10.83          25.896774   

   temp_min_observed  precip_observed        temp_source  ... era5_v10_mean  \
0           17.38424       154.966725  NYANZA LAC (IRAT)  ...      0.862369   
1           17.38900       120.119005  NYANZA LAC (IRAT)  ...      0.571162   
2           18.16450       126.911998  NYANZA LAC (IRAT)  ...      0.580950   
3           18.14420       117.792010  NYANZA LAC (IRAT)  ...      0.503440   
4           17.22097       117.313940  NYANZA LAC (IRAT)  ...      0.361469   

  era5_d2m_mean  era5_msl_mean  temp_max_imputed  

river
Buzimba       0.442345
Jiji          0.236373
Kaburantwa    0.684697
Mpanda        0.302602
Mulembwe      0.666540
Mutimbuzi     0.655716
Nyakagunda    0.087199
Nyamagana     0.267380
Nyengwe       0.385688
Rusizi        1.049724
Name: water_level, dtype: float64

In [2]:
# ============================================================
# MULTI-MODEL WATER LEVEL IMPUTATION PIPELINE
# Models:
# 1. LightGBM
# 2. XGBoost
# 3. ARIMAX with predictors
# 4. SARIMAX with predictors
# ============================================================

import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error


# ============================================================
# SETTINGS
# ============================================================

TARGET_COL = "water_level"
RIVER_COL = "river"
START_YEAR = 2000

MASK_FRACTION = 0.20
RANDOM_SEED = 42
MIN_OBS = 50


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def create_features(df, target_col=TARGET_COL):
    df = df.copy()
    df = df.sort_index()

    # Month seasonality
    df["month"] = df.index.month

    # Precipitation lags
    for lag in [1, 2, 3, 6, 12]:
        df[f"precip_lag{lag}"] = df["precip_observed"].shift(lag)

    # Rolling rainfall totals using previous months only
    df["precip_3m"] = df["precip_observed"].shift(1).rolling(3).sum()
    df["precip_6m"] = df["precip_observed"].shift(1).rolling(6).sum()
    df["precip_12m"] = df["precip_observed"].shift(1).rolling(12).sum()

    # Weighted rainfall from previous 3 months
    df["precip_weighted_3m"] = (
        df["precip_observed"].shift(1) * 0.5 +
        df["precip_observed"].shift(2) * 0.3 +
        df["precip_observed"].shift(3) * 0.2
    )

    # Temporary interpolation only for lag creation
    df["water_level_temp"] = (
        df[target_col]
        .interpolate(method="linear")
        .ffill()
        .bfill()
    )

    # Water-level lags
    for lag in [1, 2, 3, 6, 12]:
        df[f"level_lag{lag}"] = df["water_level_temp"].shift(lag)

    return df


# ============================================================
# PREDICTOR LIST
# ============================================================

predictors = [
    "temp_max_observed",
    "temp_min_observed",
    "precip_observed",
    "precip_lag1",
    "precip_lag2",
    "precip_lag3",
    "precip_lag6",
    "precip_lag12",
    "precip_3m",
    "precip_6m",
    "precip_12m",
    "precip_weighted_3m",
    "month",
    "ndvi",
    "level_lag1",
    "level_lag2",
    "level_lag3",
    "level_lag6",
    "level_lag12"
]


# ============================================================
# MACHINE LEARNING MODELS
# ============================================================

ml_models = {
    "LightGBM": LGBMRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        num_leaves=8,
        min_child_samples=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        verbose=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        objective="reg:squarederror"
    )
}


# ============================================================
# METRIC FUNCTION
# ============================================================

def calculate_metrics(y_true, y_pred, train_y):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    bias = np.mean(y_pred - y_true)
    nrmse = rmse / train_y.std() if train_y.std() != 0 else np.nan

    return rmse, mae, bias, nrmse


# ============================================================
# VALIDATION: ARTIFICIALLY MASK 20% OF OBSERVED VALUES
# ============================================================

validation_results = []
validation_predictions = []

np.random.seed(RANDOM_SEED)

for river_name in rivers[RIVER_COL].dropna().unique():

    df = rivers[rivers[RIVER_COL] == river_name].copy()
    df = df.sort_index()

    # Keep modern NDVI period
    df = df[df.index.year >= START_YEAR].copy()

    observed_idx = df[df[TARGET_COL].notna()].index

    if len(observed_idx) < MIN_OBS:
        print(f"Skipping {river_name}: not enough observed values.")
        continue

    # Artificially mask 20% of observed water levels
    masked_idx = np.random.choice(
        observed_idx,
        size=int(len(observed_idx) * MASK_FRACTION),
        replace=False
    )

    df_val = df.copy()

    true_values = df_val.loc[masked_idx, TARGET_COL].copy()
    df_val.loc[masked_idx, TARGET_COL] = np.nan

    # Create predictors after masking
    df_val = create_features(df_val, target_col=TARGET_COL)

    # Keep rows where predictors exist
    available_predictor_rows = df_val[predictors].notna().any(axis=1)

    train_df = df_val[
        df_val[TARGET_COL].notna() & available_predictor_rows
    ].copy()

    test_df = df_val.loc[masked_idx].copy()
    test_df = test_df[test_df[predictors].notna().any(axis=1)]

    if len(train_df) < MIN_OBS or len(test_df) == 0:
        print(f"Skipping {river_name}: insufficient train/test rows after feature creation.")
        continue

    y_train = train_df[TARGET_COL]
    X_train = train_df[predictors]
    X_test = test_df[predictors]
    y_test = true_values.loc[test_df.index]

    # Impute missing predictor values
    imputer = SimpleImputer(strategy="mean")
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)

    # --------------------------------------------------------
    # LightGBM and XGBoost validation
    # --------------------------------------------------------

    for model_name, model in ml_models.items():

        model.fit(X_train_imp, y_train)
        preds = model.predict(X_test_imp)

        rmse, mae, bias, nrmse = calculate_metrics(y_test, preds, y_train)

        validation_results.append({
            "river": river_name,
            "model": model_name,
            "n_train": len(train_df),
            "n_test_masked": len(test_df),
            "rmse": rmse,
            "mae": mae,
            "bias": bias,
            "nrmse": nrmse
        })

        validation_predictions.append(pd.DataFrame({
            "river": river_name,
            "model": model_name,
            "date": test_df.index,
            "true_water_level": y_test.values,
            "predicted_water_level": preds,
            "error": preds - y_test.values
        }))

    # --------------------------------------------------------
    # ARIMAX validation
    # ARIMAX = SARIMAX with exogenous predictors but no seasonal term
    # --------------------------------------------------------

    try:
        df_arimax = df_val.copy()
        df_arimax = df_arimax[df_arimax[predictors].notna().any(axis=1)].copy()

        y_arimax = df_arimax[TARGET_COL]
        X_arimax = df_arimax[predictors]

        arimax_imputer = SimpleImputer(strategy="mean")
        X_arimax_imp = arimax_imputer.fit_transform(X_arimax)

        arimax_model = SARIMAX(
            y_arimax,
            exog=X_arimax_imp,
            order=(1, 0, 1),
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        arimax_fit = arimax_model.fit(disp=False)

        arimax_pred_all = arimax_fit.predict(
            start=0,
            end=len(df_arimax) - 1,
            exog=X_arimax_imp
        )

        arimax_preds = pd.Series(
            arimax_pred_all,
            index=df_arimax.index
        ).loc[test_df.index]

        rmse, mae, bias, nrmse = calculate_metrics(y_test, arimax_preds, y_train)

        validation_results.append({
            "river": river_name,
            "model": "ARIMAX",
            "n_train": len(train_df),
            "n_test_masked": len(test_df),
            "rmse": rmse,
            "mae": mae,
            "bias": bias,
            "nrmse": nrmse
        })

        validation_predictions.append(pd.DataFrame({
            "river": river_name,
            "model": "ARIMAX",
            "date": test_df.index,
            "true_water_level": y_test.values,
            "predicted_water_level": arimax_preds.values,
            "error": arimax_preds.values - y_test.values
        }))

    except Exception as e:
        print(f"ARIMAX failed for {river_name}: {e}")

    # --------------------------------------------------------
    # SARIMAX validation
    # SARIMAX includes monthly seasonality
    # --------------------------------------------------------

    try:
        df_sarimax = df_val.copy()
        df_sarimax = df_sarimax[df_sarimax[predictors].notna().any(axis=1)].copy()

        y_sarimax = df_sarimax[TARGET_COL]
        X_sarimax = df_sarimax[predictors]

        sarimax_imputer = SimpleImputer(strategy="mean")
        X_sarimax_imp = sarimax_imputer.fit_transform(X_sarimax)

        sarimax_model = SARIMAX(
            y_sarimax,
            exog=X_sarimax_imp,
            order=(1, 0, 1),
            seasonal_order=(1, 0, 1, 12),
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        sarimax_fit = sarimax_model.fit(disp=False)

        sarimax_pred_all = sarimax_fit.predict(
            start=0,
            end=len(df_sarimax) - 1,
            exog=X_sarimax_imp
        )

        sarimax_preds = pd.Series(
            sarimax_pred_all,
            index=df_sarimax.index
        ).loc[test_df.index]

        rmse, mae, bias, nrmse = calculate_metrics(y_test, sarimax_preds, y_train)

        validation_results.append({
            "river": river_name,
            "model": "SARIMAX",
            "n_train": len(train_df),
            "n_test_masked": len(test_df),
            "rmse": rmse,
            "mae": mae,
            "bias": bias,
            "nrmse": nrmse
        })

        validation_predictions.append(pd.DataFrame({
            "river": river_name,
            "model": "SARIMAX",
            "date": test_df.index,
            "true_water_level": y_test.values,
            "predicted_water_level": sarimax_preds.values,
            "error": sarimax_preds.values - y_test.values
        }))

    except Exception as e:
        print(f"SARIMAX failed for {river_name}: {e}")


# ============================================================
# MODEL COMPARISON TABLES
# ============================================================

validation_results_df = pd.DataFrame(validation_results)
validation_predictions_df = pd.concat(validation_predictions, ignore_index=True)

# Overall model comparison
model_comparison = (
    validation_results_df
    .groupby("model")
    .agg(
        mean_rmse=("rmse", "mean"),
        median_rmse=("rmse", "median"),
        mean_mae=("mae", "mean"),
        median_mae=("mae", "median"),
        mean_bias=("bias", "mean"),
        mean_nrmse=("nrmse", "mean"),
        median_nrmse=("nrmse", "median"),
        rivers_tested=("river", "nunique")
    )
    .reset_index()
    .sort_values("mean_nrmse")
)

# Best model per river
best_model_per_river = (
    validation_results_df
    .sort_values(["river", "nrmse"])
    .groupby("river")
    .first()
    .reset_index()
)

# Count how often each model wins
model_win_counts = (
    best_model_per_river["model"]
    .value_counts()
    .reset_index()
)

model_win_counts.columns = ["model", "number_of_rivers_best"]


print("Overall model comparison:")
display(model_comparison)

print("Best model per river:")
display(best_model_per_river)

print("Number of rivers where each model performed best:")
display(model_win_counts)


# ============================================================
# SELECT OVERALL BEST MODEL
# ============================================================

overall_best_model = model_comparison.iloc[0]["model"]

print(f"Overall best model based on mean NRMSE: {overall_best_model}")






/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

Overall model comparison:


,model,mean_rmse,median_rmse,mean_mae,median_mae,mean_bias,mean_nrmse,median_nrmse,rivers_tested
3,XGBoost,0.140737,0.103586,0.106375,0.078862,-0.026182,0.336707,0.302605,10
1,LightGBM,0.142426,0.114565,0.107141,0.082304,-0.022060,0.342894,0.337111,10
0,ARIMAX,0.148757,0.117044,0.112663,0.087983,-0.018572,0.362008,0.344921,10
2,SARIMAX,0.149282,0.116541,0.112886,0.085561,-0.020515,0.364032,0.341653,10


Best model per river:


,river,model,n_train,n_test_masked,rmse,mae,bias,nrmse
0,Buzimba,SARIMAX,107,26,0.116644,0.082641,-0.028723,0.226785
1,Jiji,XGBoost,239,59,0.102261,0.069487,-0.008013,0.442203
2,Kaburantwa,XGBoost,150,37,0.140754,0.114606,-0.010223,0.239740
3,Mpanda,XGBoost,137,34,0.096407,0.075604,-0.009093,0.314945
4,Mulembwe,ARIMAX,208,51,0.273904,0.219770,-0.005326,0.442561
5,Mutimbuzi,LightGBM,86,21,0.285994,0.233052,-0.166635,0.407404
6,Nyakagunda,LightGBM,113,28,0.057375,0.044742,0.002832,0.545253
7,Nyamagana,XGBoost,138,34,0.081702,0.052202,-0.017583,0.290265
8,Nyengwe,LightGBM,110,27,0.094843,0.063780,-0.002309,0.239951
9,Rusizi,XGBoost,155,38,0.104910,0.082121,-0.004718,0.131570


Number of rivers where each model performed best:


,model,number_of_rivers_best
0,XGBoost,5
1,LightGBM,3
2,SARIMAX,1
3,ARIMAX,1


Overall best model based on mean NRMSE: XGBoost


In [3]:
rivers.head()

,river,year,month,water_level,runoff,temp_max_observed,temp_min_observed,precip_observed,temp_source,precip_source,...,era5_v10_mean,era5_d2m_mean,era5_msl_mean,temp_max_imputed,temp_min_imputed,precip_bias_factor,precip_imputed,ndvi,water_level_imputed_v2,wl_imputed_v2
date,,,,,,,,,,,,,,,,,,,,,
1981-01-01,Buzimba,1981,1,0.541000,16.94,26.619355,17.384240,154.966725,NYANZA LAC (IRAT),NYANZA LAC (IRAT),...,0.862369,17.834026,101288.167581,False,True,0.255013,True,NaN,0.541000,False
1981-01-01,Nyamagana,1981,1,0.549032,NaN,30.354839,18.038710,138.500000,MPARAMBO,MPARAMBO,...,0.607085,12.743936,101209.140181,False,False,2.186850,False,NaN,0.549032,False
1981-01-01,Nyengwe,1981,1,0.596129,13.55,26.619355,20.176480,158.824128,NYANZA LAC (IRAT),NYANZA LAC (IRAT),...,0.669341,18.981015,101215.545262,False,True,0.365854,True,NaN,0.596129,False
1981-01-01,Rusizi,1981,1,1.560000,NaN,29.148387,18.770968,73.500000,BUJUMBURA (Aeroport),BUJUMBURA (Aeroport),...,0.737239,14.708146,101196.988118,False,False,1.504363,False,NaN,1.560000,False
1981-01-01,Jiji,1981,1,0.748065,NaN,21.396774,10.119355,270.200000,MPOTA (Tora),MPOTA (Tora),...,0.444230,15.701011,101401.648401,False,False,0.746962,False,NaN,0.748065,False


In [4]:
# ============================================================
# FINAL IMPUTATION USING ALL MODELS 
# ============================================================

rivers_imputed = rivers.copy()

for model_name in ["LightGBM", "XGBoost", "ARIMAX", "SARIMAX"]:
    rivers_imputed[f"water_level_{model_name}_imputed"] = rivers_imputed[TARGET_COL]
    rivers_imputed[f"water_level_{model_name}_method"] = np.where(
        rivers_imputed[TARGET_COL].notna(),
        "observed",
        "missing"
    )


for river_name in rivers[RIVER_COL].dropna().unique():

    df = rivers[rivers[RIVER_COL] == river_name].copy()
    df = df.sort_index()
    df = df[df.index.year >= START_YEAR].copy()

    df_features = create_features(df, target_col=TARGET_COL)
    df_features = df_features[df_features[predictors].notna().any(axis=1)].copy()

    observed_df = df_features[df_features[TARGET_COL].notna()].copy()
    missing_df = df_features[df_features[TARGET_COL].isna()].copy()

    if len(observed_df) < MIN_OBS or len(missing_df) == 0:
        print(f"Skipping final imputation for {river_name}: insufficient data or no missing values.")
        continue

    X_train = observed_df[predictors]
    y_train = observed_df[TARGET_COL]
    X_missing = missing_df[predictors]

    imputer = SimpleImputer(strategy="mean")
    X_train_imp = imputer.fit_transform(X_train)
    X_missing_imp = imputer.transform(X_missing)

    target_mask = (
        (rivers_imputed[RIVER_COL] == river_name) &
        (rivers_imputed.index.isin(missing_df.index))
    )

    # --------------------------------------------------------
    # LightGBM and XGBoost final imputation
    # --------------------------------------------------------

    for model_name, model in ml_models.items():

        model.fit(X_train_imp, y_train)
        preds = model.predict(X_missing_imp)

        if target_mask.sum() != len(preds):
            print(
                f"Length mismatch for {river_name}, {model_name}: "
                f"{target_mask.sum()} rows selected but {len(preds)} predictions."
            )
            continue

        rivers_imputed.loc[
            target_mask,
            f"water_level_{model_name}_imputed"
        ] = preds

        rivers_imputed.loc[
            target_mask,
            f"water_level_{model_name}_method"
        ] = model_name

    # --------------------------------------------------------
    # ARIMAX final imputation
    # --------------------------------------------------------

    try:
        y_all = df_features[TARGET_COL]
        X_all = df_features[predictors]

        arimax_imputer = SimpleImputer(strategy="mean")
        X_all_imp = arimax_imputer.fit_transform(X_all)

        arimax_model = SARIMAX(
            y_all,
            exog=X_all_imp,
            order=(1, 0, 1),
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        arimax_fit = arimax_model.fit(disp=False)

        arimax_pred_all = arimax_fit.predict(
            start=0,
            end=len(df_features) - 1,
            exog=X_all_imp
        )

        arimax_preds = pd.Series(
            arimax_pred_all,
            index=df_features.index
        ).loc[missing_df.index]

        if target_mask.sum() != len(arimax_preds):
            print(
                f"Length mismatch for {river_name}, ARIMAX: "
                f"{target_mask.sum()} rows selected but {len(arimax_preds)} predictions."
            )
        else:
            rivers_imputed.loc[
                target_mask,
                "water_level_ARIMAX_imputed"
            ] = arimax_preds.values

            rivers_imputed.loc[
                target_mask,
                "water_level_ARIMAX_method"
            ] = "ARIMAX"

    except Exception as e:
        print(f"ARIMAX final imputation failed for {river_name}: {e}")

    # --------------------------------------------------------
    # SARIMAX final imputation
    # --------------------------------------------------------

    try:
        y_all = df_features[TARGET_COL]
        X_all = df_features[predictors]

        sarimax_imputer = SimpleImputer(strategy="mean")
        X_all_imp = sarimax_imputer.fit_transform(X_all)

        sarimax_model = SARIMAX(
            y_all,
            exog=X_all_imp,
            order=(1, 0, 1),
            seasonal_order=(1, 0, 1, 12),
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        sarimax_fit = sarimax_model.fit(disp=False)

        sarimax_pred_all = sarimax_fit.predict(
            start=0,
            end=len(df_features) - 1,
            exog=X_all_imp
        )

        sarimax_preds = pd.Series(
            sarimax_pred_all,
            index=df_features.index
        ).loc[missing_df.index]

        if target_mask.sum() != len(sarimax_preds):
            print(
                f"Length mismatch for {river_name}, SARIMAX: "
                f"{target_mask.sum()} rows selected but {len(sarimax_preds)} predictions."
            )
        else:
            rivers_imputed.loc[
                target_mask,
                "water_level_SARIMAX_imputed"
            ] = sarimax_preds.values

            rivers_imputed.loc[
                target_mask,
                "water_level_SARIMAX_method"
            ] = "SARIMAX"

    except Exception as e:
        print(f"SARIMAX final imputation failed for {river_name}: {e}")




/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warni

In [5]:
# ============================================================
# CREATE BEST-MODEL-PER-RIVER IMPUTED COLUMN
# ============================================================

# Get best model for each river based on lowest NRMSE
best_model_lookup = (
    validation_results_df
    .sort_values("nrmse")
    .groupby("river")
    .first()["model"]
    .to_dict()
)

# Create final columns
rivers_imputed["water_level_best_model_imputed"] = rivers_imputed[TARGET_COL]

rivers_imputed["best_model_used"] = np.where(
    rivers_imputed[TARGET_COL].notna(),
    "observed",
    np.nan
)

# ------------------------------------------------------------
# Fill missing values using best model for EACH river
# ------------------------------------------------------------

for river_name, best_model in best_model_lookup.items():

    river_mask = (
        (rivers_imputed[RIVER_COL] == river_name)
    )

    missing_mask = (
        river_mask &
        rivers_imputed[TARGET_COL].isna()
    )

    best_col = f"water_level_{best_model}_imputed"

    rivers_imputed.loc[
        missing_mask,
        "water_level_best_model_imputed"
    ] = rivers_imputed.loc[
        missing_mask,
        best_col
    ]

    rivers_imputed.loc[
        missing_mask,
        "best_model_used"
    ] = best_model



In [6]:
validation_results_df.head()


,river,model,n_train,n_test_masked,rmse,mae,bias,nrmse
0,Buzimba,LightGBM,107,26,0.128500,0.101118,-0.008932,0.249834
1,Buzimba,XGBoost,107,26,0.123952,0.097611,-0.025899,0.240993
2,Buzimba,ARIMAX,107,26,0.119000,0.088203,-0.015826,0.231364
3,Buzimba,SARIMAX,107,26,0.116644,0.082641,-0.028723,0.226785
4,Nyamagana,LightGBM,138,34,0.086715,0.054994,-0.006699,0.308074


In [7]:
validation_predictions_df.head()

,river,model,date,true_water_level,predicted_water_level,error
0,Buzimba,LightGBM,2004-04-01,1.567333,1.743055,0.175722
1,Buzimba,LightGBM,2005-10-01,0.472258,0.611763,0.139505
2,Buzimba,LightGBM,2002-08-01,1.157742,1.129328,-0.028414
3,Buzimba,LightGBM,2003-07-01,1.278065,1.181872,-0.096192
4,Buzimba,LightGBM,2012-09-01,0.317667,0.393878,0.076212


In [8]:
model_comparison.head()

,model,mean_rmse,median_rmse,mean_mae,median_mae,mean_bias,mean_nrmse,median_nrmse,rivers_tested
3,XGBoost,0.140737,0.103586,0.106375,0.078862,-0.026182,0.336707,0.302605,10
1,LightGBM,0.142426,0.114565,0.107141,0.082304,-0.022060,0.342894,0.337111,10
0,ARIMAX,0.148757,0.117044,0.112663,0.087983,-0.018572,0.362008,0.344921,10
2,SARIMAX,0.149282,0.116541,0.112886,0.085561,-0.020515,0.364032,0.341653,10


In [9]:
best_model_per_river.head(20)

,river,model,n_train,n_test_masked,rmse,mae,bias,nrmse
0,Buzimba,SARIMAX,107,26,0.116644,0.082641,-0.028723,0.226785
1,Jiji,XGBoost,239,59,0.102261,0.069487,-0.008013,0.442203
2,Kaburantwa,XGBoost,150,37,0.140754,0.114606,-0.010223,0.239740
3,Mpanda,XGBoost,137,34,0.096407,0.075604,-0.009093,0.314945
4,Mulembwe,ARIMAX,208,51,0.273904,0.219770,-0.005326,0.442561
5,Mutimbuzi,LightGBM,86,21,0.285994,0.233052,-0.166635,0.407404
6,Nyakagunda,LightGBM,113,28,0.057375,0.044742,0.002832,0.545253
7,Nyamagana,XGBoost,138,34,0.081702,0.052202,-0.017583,0.290265
8,Nyengwe,LightGBM,110,27,0.094843,0.063780,-0.002309,0.239951
9,Rusizi,XGBoost,155,38,0.104910,0.082121,-0.004718,0.131570


In [10]:
model_win_counts.head()

,model,number_of_rivers_best
0,XGBoost,5
1,LightGBM,3
2,SARIMAX,1
3,ARIMAX,1


In [11]:
# Keep only years from 2000 onward
rivers_imputed = rivers_imputed[
    rivers_imputed.index.year >= 2000
].copy()
rivers_imputed.head()

,river,year,month,water_level,runoff,temp_max_observed,temp_min_observed,precip_observed,temp_source,precip_source,...,water_level_LightGBM_imputed,water_level_LightGBM_method,water_level_XGBoost_imputed,water_level_XGBoost_method,water_level_ARIMAX_imputed,water_level_ARIMAX_method,water_level_SARIMAX_imputed,water_level_SARIMAX_method,water_level_best_model_imputed,best_model_used
date,,,,,,,,,,,,,,,,,,,,,
2000-01-01,Nyamagana,2000,1,NaN,NaN,33.504850,15.512020,30.170508,MPARAMBO,MPARAMBO,...,0.745182,LightGBM,0.737020,XGBoost,0.708845,ARIMAX,0.708845,SARIMAX,0.737020,XGBoost
2000-01-01,Jiji,2000,1,0.704516,NaN,21.448387,9.948387,125.600000,MPOTA (Tora),MPOTA (Tora),...,0.704516,observed,0.704516,observed,0.704516,observed,0.704516,observed,0.704516,observed
2000-01-01,Kaburantwa,2000,1,NaN,0.71,33.504850,15.512020,30.459723,MPARAMBO,MPARAMBO,...,1.541039,LightGBM,1.531842,XGBoost,1.195061,ARIMAX,1.195061,SARIMAX,1.531842,XGBoost
2000-01-01,Mulembwe,2000,1,1.239355,20.05,21.448387,9.948387,125.600000,MPOTA (Tora),MPOTA (Tora),...,1.239355,observed,1.239355,observed,1.239355,observed,1.239355,observed,1.239355,observed
2000-01-01,Nyengwe,2000,1,NaN,12.99,28.367742,18.761290,203.000000,NYANZA LAC (IRAT),NYANZA LAC (IRAT),...,0.890016,LightGBM,0.928992,XGBoost,0.903728,ARIMAX,0.903728,SARIMAX,0.890016,LightGBM


In [12]:
rivers_imputed[
    rivers_imputed["water_level_LightGBM_method"] != "observed"
][[
    RIVER_COL,
    "water_level_LightGBM_imputed",
    "water_level_LightGBM_method",
    "water_level_XGBoost_imputed",
    "water_level_XGBoost_method",
    "water_level_ARIMAX_imputed",
    "water_level_ARIMAX_method",
    "water_level_SARIMAX_imputed",
    "water_level_SARIMAX_method", 
    "water_level_best_model_imputed",
    "best_model_used" 

]].head(20)

,river,water_level_LightGBM_imputed,water_level_LightGBM_method,water_level_XGBoost_imputed,water_level_XGBoost_method,water_level_ARIMAX_imputed,water_level_ARIMAX_method,water_level_SARIMAX_imputed,water_level_SARIMAX_method,water_level_best_model_imputed,best_model_used
date,,,,,,,,,,,
2000-01-01,Nyamagana,0.745182,LightGBM,0.737020,XGBoost,0.708845,ARIMAX,0.708845,SARIMAX,0.737020,XGBoost
2000-01-01,Kaburantwa,1.541039,LightGBM,1.531842,XGBoost,1.195061,ARIMAX,1.195061,SARIMAX,1.531842,XGBoost
2000-01-01,Nyengwe,0.890016,LightGBM,0.928992,XGBoost,0.903728,ARIMAX,0.903728,SARIMAX,0.890016,LightGBM
2000-01-01,Mutimbuzi,1.575930,LightGBM,1.537429,XGBoost,1.260306,ARIMAX,1.260306,SARIMAX,1.575930,LightGBM
2000-01-01,Rusizi,2.834729,LightGBM,2.802466,XGBoost,2.480451,ARIMAX,2.479945,SARIMAX,2.802466,XGBoost
2000-01-01,Nyakagunda,0.497807,LightGBM,0.507216,XGBoost,0.481142,ARIMAX,0.481142,SARIMAX,0.497807,LightGBM
2000-02-01,Nyengwe,0.919100,LightGBM,0.933955,XGBoost,0.934741,ARIMAX,0.934741,SARIMAX,0.919100,LightGBM
2000-02-01,Mutimbuzi,1.128504,LightGBM,0.986803,XGBoost,0.888987,ARIMAX,0.888987,SARIMAX,1.128504,LightGBM
2000-02-01,Kaburantwa,0.948949,LightGBM,0.955372,XGBoost,0.778587,ARIMAX,0.778587,SARIMAX,0.955372,XGBoost


In [13]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

#display(rivers_imputed)

# threshold 

In [18]:
nrmse_table = (
    validation_results_df
    .pivot(
        index="river",
        columns="model",
        values="nrmse"
    )
)

nrmse_table

model,ARIMAX,LightGBM,SARIMAX,XGBoost
river,,,,
Buzimba,0.231364,0.249834,0.226785,0.240993
Jiji,0.497668,0.443148,0.503501,0.442203
Kaburantwa,0.254481,0.246476,0.254913,0.239740
Mpanda,0.361406,0.366148,0.351924,0.314945
Mulembwe,0.442561,0.475861,0.444711,0.475234
Mutimbuzi,0.473341,0.407404,0.479238,0.438884
Nyakagunda,0.616619,0.545253,0.632434,0.546038
Nyamagana,0.328436,0.308074,0.331383,0.290265
Nyengwe,0.249559,0.239951,0.249645,0.247203


In [19]:
nrmse_table["best_nrmse"] = nrmse_table.min(axis=1)

for model in ["LightGBM","XGBoost","ARIMAX","SARIMAX"]:
    nrmse_table[f"{model}_delta"] = (
        nrmse_table[model]
        - nrmse_table["best_nrmse"]
    )

nrmse_table

model,ARIMAX,LightGBM,SARIMAX,XGBoost,best_nrmse,LightGBM_delta,XGBoost_delta,ARIMAX_delta,SARIMAX_delta
river,,,,,,,,,
Buzimba,0.231364,0.249834,0.226785,0.240993,0.226785,0.023049,0.014208,0.004579,0.000000
Jiji,0.497668,0.443148,0.503501,0.442203,0.442203,0.000945,0.000000,0.055465,0.061298
Kaburantwa,0.254481,0.246476,0.254913,0.239740,0.239740,0.006736,0.000000,0.014741,0.015173
Mpanda,0.361406,0.366148,0.351924,0.314945,0.314945,0.051203,0.000000,0.046461,0.036979
Mulembwe,0.442561,0.475861,0.444711,0.475234,0.442561,0.033300,0.032673,0.000000,0.002150
Mutimbuzi,0.473341,0.407404,0.479238,0.438884,0.407404,0.000000,0.031480,0.065937,0.071835
Nyakagunda,0.616619,0.545253,0.632434,0.546038,0.545253,0.000000,0.000785,0.071366,0.087181
Nyamagana,0.328436,0.308074,0.331383,0.290265,0.290265,0.017809,0.000000,0.038171,0.041118
Nyengwe,0.249559,0.239951,0.249645,0.247203,0.239951,0.000000,0.007252,0.009608,0.009694


In [ ]:
# threshold 

THRESHOLD = 0.02

lightgbm_close = (
    nrmse_table["LightGBM_delta"]
    <= THRESHOLD
)

lightgbm_close.sum()

# LightGBM was within 0.02 NRMSE of the best-performing model for 70% of rivers. Indication that it could be suitable to use one model for all rivers. 
# but we need to check the other 30% and are there specific cases where we should use a different model...

7

# prediction 